# IMPORTS

In [1]:
from qiskit import *
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_state_city
import qiskit.quantum_info as qi
from qiskit.visualization import plot_histogram
from math import gcd , ceil
from numpy.random import randint
import pandas as pd
from fractions import Fraction
from tqdm import tqdm
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
import matplotlib.pyplot as plt
import random
import sys
import math
import array
import fractions
import numpy as np

In [2]:
from qiskit.circuit.library import QFT, UnitaryGate
from tqdm import tqdm

# Manually calculate period

In [3]:
def period(a,N): # returns r
    new = a
    i = 1
    while new != 1 and i < N:
        new = (new * a) % N
        i+=1
    return i

In [4]:
def modinv(a,N): # returns a^(-1)
    r = period(a,N)
    return (a ** (r-1)) % N

# Defining the oracle

In [5]:
def f_shor(x, a, N):
    return pow(a, x, N)

In [6]:
def shor_oracle(a, N, qubits_state, qubits_eval):
    n_qubits = qubits_eval + qubits_state
    outputs = np.zeros(2 ** n_qubits, dtype=int)

    for state in tqdm(range(2 ** n_qubits)):
        state_bin = format(state, f"0{n_qubits}b")

        y_bin = state_bin[:qubits_state]
        x_bin = state_bin[qubits_state:]

        x = int(x_bin, 2)
        y = int(y_bin, 2)

        fx = f_shor(x, a, N)
        y_xor_fx = fx ^ y

        y_out_bin = format(y_xor_fx, f"0{qubits_state}b")
        output_bin = y_out_bin + x_bin

        outputs[state] = int(output_bin, 2)

    U = np.zeros((2 ** n_qubits, 2 ** n_qubits), dtype=int)

    for y in tqdm(range(2 ** n_qubits)):
        for i in range(2 ** n_qubits):
            U[i][y] = 1 if outputs[y] == i else 0

    return UnitaryGate(U, label=f'{a}^x mod {N}')

# Making Circuit for Factoring

In [23]:
def shor_circuit(a, N):
    n = ceil(np.log2(N))
    e_qb = 2 * n
    s_qb = n

    qr_x = QuantumRegister(e_qb, 'x')
    qr_y = QuantumRegister(s_qb, 'y')
    cr = ClassicalRegister(e_qb, 'c')

    qc = QuantumCircuit(qr_x, qr_y, cr)
    qc.h(qr_x)
    qc.barrier()

    oracle = shor_oracle(a, N, s_qb, e_qb)
    qc.append(oracle, range(e_qb + s_qb))

    qc.barrier()
    inv_qft = QFT(e_qb, inverse=True).to_gate()
    qc.append(inv_qft, qr_x)
    print(qc)
    return qc, e_qb

In [20]:
def eval_shor(qc, a, N, e_qb):
    qc.measure(range(e_qb), range(e_qb))

    backend = AerSimulator()
    job = backend.run(transpile(qc, backend), shots=32, memory=True)
    memory = job.result().get_memory()

    for readout in memory:
        phase = int(readout, 2) / (2 ** e_qb)

        frac = Fraction(phase).limit_denominator(N)
        r_candidate = frac.denominator

        if r_candidate % 2 != 0:
            continue

        # Check if valid period
        if pow(a, r_candidate, N) == 1:
            return r_candidate

    return None

In [21]:
def run_shor(a, N):
    qc, e_qb = shor_circuit(a, N)
    r = eval_shor(qc, a, N, e_qb)

    if r is None:
        return None

    if r % 2 != 0:
        return None

    x = pow(a, r // 2, N)

    factor1 = gcd(x - 1, N)
    factor2 = gcd(x + 1, N)

    if factor1 in [1, N]:
        return None

    return factor1, factor2


In [ ]:
print(run_shor(5,21))